## 1. Verify GPU

In [ ]:
/
import torch
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert 'A100' in torch.cuda.get_device_name(0), 'WARNING: Not an A100 - need to change runtimes to aquire for training'      


## 2. Mount Google Drive (checkpoint persistence)

In [ ]:
/
from google.colab import drive
drive.mount('/content/drive)     

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/vektor-guard/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkppoint dir: {CHECKPOINT_DIR}')


## 3. Clone repo and install dependencies

In [ ]:
import os
if not os.path.exists('/content/vektor'):
    !git clone https://github.com/emsikes/vektor.git /content/vektor
%cd /content/vektor/platform
!pip install -r requirements.txt -q


## 4. Authenticate HuggingFace and WandB

In [ ]:
from huggingface_hub import login as hf_login
from google.colab import userdata
import wandb

# secrets stored in Colab Secrets (left sidebar → key icon)
hf_login(token=userdata.get('HF_TOKEN'))
wandb.login(key=userdata.get('WANDB_API_KEY'))


## 5. Upload data splits

In [ ]:
# upload train.json, val.json, test.json from your local machine
# Runtime → upload files, or copy from Drive if already uploaded
import os
os.makedirs('data/splits', exist_ok=True)
from google.colab import files
print('Upload train.json, val.json, test.json when prompted')
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f'data/splits/{fname}', 'wb') as f:
        f.write(data)
    print(f'Saved data/splits/{fname}')


## 6. Train

In [ ]:
import sys
sys.path.insert(0, '/content/vektor/platform')

from src.training.trainer import build_trainer

# point output_dir to Drive so checkpoints survive session expiry
trainer = build_trainer()
trainer.args.output_dir = CHECKPOINT_DIR

trainer.train()


## 7. Evaluate on test set

In [ ]:
from src.training.dataset import load_split, build_tokenizer, tokenize_split
from src.training.metrics import compute_metrics, check_targets
import numpy as np

config_model = 'answerdotai/ModernBERT-large'
tokenizer = build_tokenizer(config_model)
test_dataset = tokenize_split(load_split('test'), tokenizer)

# run inference on test set using best checkpoint
predictions = trainer.predict(test_dataset)
metrics = compute_metrics((predictions.predictions, predictions.label_ids))

print(metrics)
check_targets(metrics)


## 8. Push best model to HuggingFace Hub

In [ ]:
trainer.model.push_to_hub('theinferenceloop/vektor-guard-v1')
tokenizer.push_to_hub('theinferenceloop/vektor-guard-v1')
print('Model pushed to HuggingFace Hub')
